# Noisy Heisenberg rho plots

Read `data/noisy_heisenberg_* / metadata.json` and generate publication-style error-scaling PDFs.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Match the existing plotting notebooks in this project.
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams.update({"font.size": 18})
plt.rcParams["text.usetex"] = True
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "plot" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "plot" / "noisy_heisenberg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NOISES = ["amplitude_damping", "dephasing"]
NOISE_TITLES = {
    "amplitude_damping": "local amplitude damping",
    "dephasing": "local dephasing",
}
INITIALS = ["0", "1", "+", "I"]
INITIAL_TAGS = {"0": "000", "1": "111", "+": "+++", "I": "mixed"}
INITIAL_TITLES = {
    "0": r"$\rho_0=|0^N\rangle\langle 0^N|$",
    "1": r"$\rho_0=|1^N\rangle\langle 1^N|$",
    "+": r"$\rho_0=|+^N\rangle\langle +^N|$",
    "I": r"$\rho_0=I^{\otimes N}/2^N$",
}
COLORS = {4: "#1f77b4", 6: "#ff7f0e", 8: "#2ca02c", 10: "#d62728"}
MARKERS = {0.1: "o", 1.0: "D"}
LINESTYLES = {0.1: "-", 1.0: "--"}
FIT_ERROR_FLOOR = 1e-12


def load_metadata(noise):
    path = DATA_ROOT / f"noisy_heisenberg_{noise}" / "metadata.json"
    with path.open() as f:
        return json.load(f)


def fit_rows(rows):
    slopes = {}
    gammas = sorted({row["gamma"] for row in rows})
    r_list = sorted({row["r"] for row in rows})
    for gamma in gammas:
        for r in r_list:
            selected = [
                row
                for row in rows
                if row["gamma"] == gamma and row["r"] == r and row["trace_error"] > FIT_ERROR_FLOOR
            ]
            if len(selected) < 2:
                continue
            selected = sorted(selected, key=lambda row: row["N"])
            logN = np.log10([row["N"] for row in selected])
            logerr = np.log10([row["trace_error"] for row in selected])
            slope, intercept = np.polyfit(logN, logerr, 1)
            slopes[(gamma, r)] = (float(slope), float(intercept))
    return slopes


def format_panel_axes(ax, state_rows, show_ylabel=True):
    N_list = sorted({row["N"] for row in state_rows})
    logN = np.log10(N_list)
    xpad = 0.03 * (logN[-1] - logN[0])
    ax.set_xlim(logN[0] - xpad, logN[-1] + xpad)
    ax.set_xticks(np.log10(N_list))
    ax.set_xticklabels([str(N) for N in N_list])
    ax.tick_params(axis="both", labelsize=18)
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("qubit number $N$", fontsize=24)
    if show_ylabel:
        ax.set_ylabel(r"$\lg \|(e^{t\mathcal{L}}-\mathcal{S}(t/r)^r)\rho_0\|_1$", fontsize=21)
    ax.set_title(f"Lindbladian simulation error vs. system size $N$", fontsize=24)


def draw_panel(ax, rows, initial, show_ylabel=True):
    state_rows = [row for row in rows if row["initial"] == initial]
    slopes = fit_rows(state_rows)
    if not slopes:
        positive_errors = [row["trace_error"] for row in state_rows if row["trace_error"] > 0]
        if positive_errors:
            log_errors = np.log10(positive_errors)
            ymin = np.floor(np.min(log_errors) * 2) / 2 - 0.1
            ymax = np.ceil(np.max(log_errors) * 2) / 2 + 0.1
            if ymax <= ymin:
                ymax = ymin + 1.0
            ax.set_ylim(ymin, ymax)
        else:
            ax.set_ylim(-15.5, -13.0)
        format_panel_axes(ax, state_rows, show_ylabel=show_ylabel)
        ax.text(
            0.5,
            0.5,
            "zero within\nnumerical precision",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=20,
        )
        return slopes

    for gamma in sorted({row["gamma"] for row in state_rows}):
        for r in sorted({row["r"] for row in state_rows}):
            if (gamma, r) not in slopes:
                continue
            selected = [
                row
                for row in state_rows
                if row["gamma"] == gamma and row["r"] == r and row["trace_error"] > FIT_ERROR_FLOOR
            ]
            selected = sorted(selected, key=lambda row: row["N"])
            N_values = np.array([row["N"] for row in selected])
            logN = np.log10(N_values)
            logerr = np.log10([row["trace_error"] for row in selected])
            slope, intercept = slopes[(gamma, r)]
            ax.plot(
                logN,
                logerr,
                marker=MARKERS[gamma],
                color=COLORS[r],
                linestyle="",
                markersize=6,
            )
            ax.plot(
                logN,
                slope * logN + intercept,
                color=COLORS[r],
                linestyle=LINESTYLES[gamma],
                linewidth=1.6,
            )

    format_panel_axes(ax, state_rows, show_ylabel=show_ylabel)
    return slopes


def legend_handles(slopes=None):
    if slopes is None:
        r_handles = [Line2D([0], [0], color=COLORS[r], lw=2, label=fr"$r={r}$") for r in sorted(COLORS)]
        gamma_handles = [
            Line2D([0], [0], color="0.25", marker="o", linestyle="-", lw=2, label=r"$\gamma=0.1$"),
            Line2D([0], [0], color="0.25", marker="D", linestyle="--", lw=2, label=r"$\gamma=1$"),
        ]
        return r_handles + gamma_handles

    handles = []
    for gamma, r in sorted(slopes):
        slope, _ = slopes[(gamma, r)]
        handles.append(
            Line2D(
                [0],
                [0],
                color=COLORS[r],
                marker=MARKERS[gamma],
                linestyle=LINESTYLES[gamma],
                label=fr"$\gamma={gamma:.1f}, r={r}$, slope={slope:.3f}",
            )
        )
    return handles


def save_single_state_plot(data, initial):
    rows = data["rows"]
    noise = data["parameters"]["noise"]
    fig, ax = plt.subplots(figsize=(12.5, 6))
    slopes = draw_panel(ax, rows, initial)
    handles = legend_handles(slopes)
    legend_kwargs = dict(
        title=INITIAL_TITLES[initial],
        title_fontsize=20,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=20,
    )
    if handles:
        ax.legend(handles=handles, **legend_kwargs)
    else:
        blank_handles = []
        blank_labels = []
        for gamma in sorted(MARKERS):
            for r in sorted(COLORS):
                blank_handles.append(
                    Line2D(
                        [0],
                        [0],
                        color="white",
                        marker=MARKERS[gamma],
                        linestyle=LINESTYLES[gamma],
                    )
                )
                blank_labels.append(fr"$\gamma={gamma:.1f}, r={r}$, slope=0.924")
        legend = ax.legend(
            handles=blank_handles,
            labels=blank_labels,
            **legend_kwargs,
        )
        for text in legend.get_texts():
            text.set_color("white")
    fig.tight_layout()
    out = OUT_DIR / f"error-N-{INITIAL_TAGS[initial]}-heisenberg-{noise}.pdf"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    return out, slopes


all_outputs = []
all_slopes = {}
for noise in NOISES:
    data = load_metadata(noise)
    all_slopes[noise] = {}
    for initial in data["parameters"]["initials"]:
        out, slopes = save_single_state_plot(data, initial)
        all_outputs.append(str(out.relative_to(PROJECT_ROOT)))
        for (gamma, r), (slope, _) in slopes.items():
            all_slopes[noise][f"initial={initial}, gamma={gamma:.1f}, r={r}"] = slope

(OUT_DIR / "fit_slopes.json").write_text(json.dumps(all_slopes, indent=2))
print("Wrote figures:")
for output in all_outputs:
    print("  " + output)
print("Wrote " + str((OUT_DIR / "fit_slopes.json").relative_to(PROJECT_ROOT)))